In [1]:
from ati.tools.reader.lidar_pb import LidarSmallPb
import os
import plotly.graph_objects as go
import numpy as np


### loading the lidar file

In [2]:
srcdir = os.getcwd()

index_path = os.path.join(srcdir, "debug", "lidar-index.csv")
if not os.path.exists(index_path):
    print("Missing:", index_path)

In [3]:
try: 
    pb = LidarSmallPb(srcdir)
except FileNotFoundError as e:
    print("couldnt open lidar pb:", e)


print(pb.num_frames())

7461


In [4]:
start_frame = 1900 ; end_frame = 3300


### Building the visualiser

In [5]:
from ati.core.perception.sensors.lidar.lidar_frame_params import *
import open3d as o3d
import time
from matplotlib import cm

def plot_lidar_3d(pb, start_frame, end_frame, step=1, colour_col=F_RANGE, point_size=1.0,
                  cmax=15, fps=10):
    vis = o3d.visualization.Visualizer()
    vis.create_window(window_name="lidar", width=1280, height=720)
    opt = vis.get_render_option()
    opt.point_size = point_size
    opt.background_color = np.array([0.0, 0.0, 0.0])

    pcd = o3d.geometry.PointCloud()
    added = False
    try:
        fid = start_frame
        while fid <= end_frame:
            t0 = time.time()
            ts, frame = pb.get_frame(fid)
            pts = frame[frame[:, F_RANGE] > 0.1]

            pcd.points = o3d.utility.Vector3dVector(pts[:, [F_X, F_Y, F_Z]].astype(np.float64))
            c = np.clip(pts[:, colour_col] / cmax, 0, 1)  # fixed colour scale 0..cmax
            pcd.colors = o3d.utility.Vector3dVector(cm.viridis(c)[:, :3])

            if not added:
                vis.add_geometry(pcd)  # first frame also sets the initial camera
                added = True
            else:
                vis.update_geometry(pcd)

            if not vis.poll_events():  # window was closed
                break
            vis.update_renderer()
            print(f"frame {fid}  ts {ts}", end="\r")

            fid += step
            time.sleep(max(0.0, 1.0 / fps - (time.time() - t0)))
    finally:
        vis.destroy_window()

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [6]:
fig = plot_lidar_3d(pb, start_frame, end_frame, step=1)

### onto icp with the lidar data

In [6]:
import kiss_icp
print(kiss_icp.__version__)


import genz_icp
print(genz_icp.__version__)



1.2.3
0.3.2


In [7]:
def plot_lidar_odom_3d(pb, fids, poses, colour_col=F_RANGE, point_size=1.0, cmax=15, fps=10,
                       follow=False, grid_size=20, grid_spacing=1.0):
    # odom frame = pose of the first plotted frame, so every run starts at the origin
    poses = np.linalg.inv(poses[0]) @ poses

    vis = o3d.visualization.Visualizer()
    vis.create_window(window_name="lidar + odom", width=1280, height=720)
    opt = vis.get_render_option()
    opt.point_size = point_size
    opt.background_color = np.array([0.0, 0.0, 0.0])

    # static odom frame axes + ground grid at z=0 (lidar height at the start)
    odom_axes = o3d.geometry.TriangleMesh.create_coordinate_frame(size=2.0)
    ticks = np.arange(-grid_size, grid_size + grid_spacing, grid_spacing)
    grid_pts = []
    for v in ticks:
        grid_pts += [[v, -grid_size, 0], [v, grid_size, 0], [-grid_size, v, 0], [grid_size, v, 0]]
    grid = o3d.geometry.LineSet(
        points=o3d.utility.Vector3dVector(grid_pts),
        lines=o3d.utility.Vector2iVector([[j, j + 1] for j in range(0, len(grid_pts), 2)]))
    grid.paint_uniform_color([0.3, 0.3, 0.3])

    pcd = o3d.geometry.PointCloud()
    traj = o3d.geometry.LineSet()
    axes = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0)
    axes_verts = np.asarray(axes.vertices).copy()  # robot-frame axes, re-posed every frame

    try:
        for i, (fid, pose) in enumerate(zip(fids, poses)):
            t0 = time.time()
            ts, frame = pb.get_frame(fid)
            pts = frame[frame[:, F_RANGE] > 0.1]
            R, t = pose[:3, :3], pose[:3, 3]

            # lidar frame -> world frame
            xyz = pts[:, [F_X, F_Y, F_Z]].astype(np.float64)
            pcd.points = o3d.utility.Vector3dVector(xyz @ R.T + t)
            c = np.clip(pts[:, colour_col] / cmax, 0, 1)
            pcd.colors = o3d.utility.Vector3dVector(cm.viridis(c)[:, :3])

            # trajectory so far, as a red polyline
            traj.points = o3d.utility.Vector3dVector(poses[:i + 1, :3, 3])
            traj.lines = o3d.utility.Vector2iVector(
                np.column_stack([np.arange(i), np.arange(1, i + 1)]).astype(np.int32))
            traj.paint_uniform_color([1.0, 0.0, 0.0])

            # current robot pose
            axes.vertices = o3d.utility.Vector3dVector(axes_verts @ R.T + t)

            if i == 0:
                for g in (grid, odom_axes, pcd, traj, axes):
                    vis.add_geometry(g)
                # camera fixed on the odom origin, behind the start and looking down at it
                vc = vis.get_view_control()
                vc.set_lookat([0.0, 0.0, 0.0])
                vc.set_up([0.0, 0.0, 1.0])
                vc.set_front([0.0, -1.0, 1.0])
                vc.set_zoom(0.3)
            else:
                for g in (pcd, traj, axes):
                    vis.update_geometry(g)
            if follow:
                vis.get_view_control().set_lookat(t)

            if not vis.poll_events():  # window was closed
                break
            vis.update_renderer()
            print(f"frame {fid}  xyz {np.round(t, 2)}", end="\r")

            time.sleep(max(0.0, 1.0 / fps - (time.time() - t0)))
    finally:
        vis.destroy_window()

In [8]:
def run_kiss_icp(pb, config, start_frame, end_frame):
    kiss_odom = KissICP(config)  # fresh local map every run
    fids, stamps, poses = [], [], []
    t0 = time.time()
    for fid in range(start_frame, end_frame + 1):
        ts, frame = pb.get_frame(fid)
        if ts == -1:  # frame id not in the index
            continue
        pts = frame[frame[:, F_RANGE] > 0.1][:, [F_X, F_Y, F_Z]].astype(np.float64)
        kiss_odom.register_frame(pts, np.zeros(len(pts))) 

        fids.append(fid)
        stamps.append(ts)
        poses.append(kiss_odom.last_pose.copy())

        if len(fids) % 10 == 0:
            print(f"frame {fid}  xyz {np.round(poses[-1][:3, 3], 3)}")

    n = len(fids)
    print(f"{n} frames in {time.time() - t0:.1f}s ({n / (time.time() - t0):.1f} fps)")
    return np.array(fids), np.array(stamps), np.array(poses)

In [10]:
def run_genz_icp(pb, config, start_frame, end_frame):
    genz_odom = GenZICP(config)  # config is copied into C++ here, set it before calling
    fids, stamps, poses = [], [], []
    t0 = time.time()
    for fid in range(start_frame, end_frame + 1):
        ts, frame = pb.get_frame(fid)
        if ts == -1:  # frame id not in the index
            continue
        pts = frame[frame[:, F_RANGE] > 0.1][:, [F_X, F_Y, F_Z]].astype(np.float64)
        genz_odom.register_frame(pts, None)  # None -> no-timestamp path

        fids.append(fid)
        stamps.append(ts)
        poses.append(genz_odom.last_pose.copy())

        if len(fids) % 10 == 0:
            print(f"frame {fid}  xyz {np.round(poses[-1][:3, 3], 3)}")

    n = len(fids)
    print(f"{n} frames in {time.time() - t0:.1f}s ({n / (time.time() - t0):.1f} fps)")
    return np.array(fids), np.array(stamps), np.array(poses)


In [13]:
from kiss_icp.config import load_config
from kiss_icp.kiss_icp import KissICP
config = load_config(None, max_range=30.0)
config.data.deskew = False
config.data.min_range = 1.0

print(config)

out_dir='results' data=DataConfig(max_range=30.0, min_range=1.0, deskew=False) registration=RegistrationConfig(max_num_iterations=500, convergence_criterion=0.0001, max_num_threads=0) mapping=MappingConfig(voxel_size=0.3, max_points_per_voxel=20) adaptive_threshold=AdaptiveThresholdConfig(fixed_threshold=None, initial_threshold=2.0, min_motion_th=0.1)


In [14]:
fids, stamps, poses = run_kiss_icp(pb, config, start_frame, end_frame)

frame 1909  xyz [-0. -0. -0.]
frame 1919  xyz [-0.    -0.001 -0.   ]
frame 1929  xyz [0.001 0.    0.   ]
frame 1939  xyz [-0.001  0.    -0.   ]
frame 1949  xyz [-0.001  0.     0.   ]
frame 1959  xyz [ 0.001 -0.     0.   ]
frame 1969  xyz [-0.001  0.     0.   ]
frame 1979  xyz [-0.001  0.    -0.   ]
frame 1989  xyz [-0.     0.001 -0.   ]
frame 1999  xyz [-0. -0. -0.]
frame 2009  xyz [-0.001  0.    -0.   ]
frame 2019  xyz [-0.001 -0.    -0.   ]
frame 2029  xyz [-0.001  0.    -0.   ]
frame 2039  xyz [ 0.    -0.001 -0.   ]
frame 2049  xyz [-0.  0. -0.]
frame 2059  xyz [ 0.  0. -0.]
frame 2069  xyz [-0.     0.001  0.   ]
frame 2079  xyz [-0.  0.  0.]
frame 2089  xyz [-0.001  0.    -0.   ]
frame 2099  xyz [-0.001 -0.    -0.   ]
frame 2109  xyz [ 0.001 -0.001 -0.   ]
frame 2119  xyz [ 0. -0. -0.]
frame 2129  xyz [-0.001  0.    -0.   ]
frame 2139  xyz [0. 0. 0.]
frame 2149  xyz [-0.002 -0.     0.   ]
frame 2159  xyz [-0.002 -0.    -0.   ]
frame 2169  xyz [-0.03   0.084  0.001]
frame 2179  xyz 

In [15]:
plot_lidar_odom_3d(pb, fids, poses, follow=True)

[Open3D WARNING] [SimpleShaderForLineSet] Binding failed with empty geometry::LineSet.
[Open3D WARNING] [SimpleShaderForLineSet] Binding failed when preparing data.
[Open3D WARNING] [SimpleShaderForLineSet] Something is wrong in compiling or binding.


In [16]:
from pathlib import Path
from genz_icp.config import load_config as load_genz_config  
from genz_icp.genz_icp import GenZICP
preset = Path(genz_icp.__file__).parent / "config" / "pretuned" / "corridor.yaml"
genz_config = load_genz_config(preset)
genz_config.data.max_range = 30.0
genz_config.data.min_range = 1.0  

print(genz_config)

out_dir='results' data=DataConfig(max_range=30.0, min_range=1.0, deskew=False) registration=RegistrationConfig(max_num_iterations=100, convergence_criterion=0.0001, max_num_threads=0) mapping=MappingConfig(voxel_size=0.2, map_cleanup_radius=100.0, desired_num_voxelized_points=2000, planarity_threshold=0.2, max_points_per_voxel=1) adaptive_threshold=AdaptiveThresholdConfig(fixed_threshold=None, initial_threshold=2.0, min_motion_th=0.1)


In [23]:
genz_fids, genz_stamps, genz_poses = run_genz_icp(pb, genz_config, start_frame, end_frame)

frame 1909  xyz [ 0.001 -0.001  0.001]
frame 1919  xyz [-0.004  0.001  0.002]
frame 1929  xyz [-0.001  0.003  0.002]
frame 1939  xyz [0.    0.002 0.002]
frame 1949  xyz [-0.004  0.004  0.002]
frame 1959  xyz [ 0.001 -0.002  0.001]
frame 1969  xyz [-0.001  0.003  0.001]
frame 1979  xyz [0.    0.002 0.001]
frame 1989  xyz [-0.002  0.005  0.002]
frame 1999  xyz [-0.002 -0.002  0.   ]
frame 2009  xyz [-0.    -0.002  0.001]
frame 2019  xyz [ 0.001 -0.002  0.002]
frame 2029  xyz [ 0.001 -0.002  0.002]
frame 2039  xyz [-0.001 -0.001  0.002]
frame 2049  xyz [ 0.003 -0.001  0.001]
frame 2059  xyz [ 0.001 -0.002  0.002]
frame 2069  xyz [ 0.001 -0.001  0.001]
frame 2079  xyz [ 0.003 -0.001  0.002]
frame 2089  xyz [-0.002 -0.     0.002]
frame 2099  xyz [-0.002 -0.001  0.001]
frame 2109  xyz [0.    0.001 0.001]
frame 2119  xyz [-0.     0.003  0.001]
frame 2129  xyz [-0.002  0.001  0.001]
frame 2139  xyz [0.002 0.002 0.001]
frame 2149  xyz [-0.002  0.001  0.001]
frame 2159  xyz [-0.001 -0.001  0.001

In [22]:
plot_lidar_odom_3d(pb, genz_fids, genz_poses, follow=True)

[Open3D WARNING] [SimpleShaderForLineSet] Binding failed with empty geometry::LineSet.
[Open3D WARNING] [SimpleShaderForLineSet] Binding failed when preparing data.
[Open3D WARNING] [SimpleShaderForLineSet] Something is wrong in compiling or binding.


KeyboardInterrupt: 